# Velocity & Backlog Projection

Velocity tracking for GitHub Project
[#2](https://github.com/orgs/greenearth-social/projects/2/views/13), based on the
**Backlog - Unified** view — all remaining, non-Icebox work (Backlog, In
Progress, In Review, On Hold) across every repo.

All computation lives in the `velocity` library (unit-tested); this notebook is
presentation only.

**Prerequisite:** the `gh` CLI must be authenticated with the project scope:

```
gh auth refresh -s read:project
```

**Known limitation:** the GitHub GraphQL API returns project items in the
board's manual order, which we treat as priority order. Per-view (view #13)
custom sorting is not exposed by the API.

In [ ]:
# Auto-reload the `velocity` library on edit, so you don't have to
# restart the kernel after changing library code.
%load_ext autoreload
%autoreload 2

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # repo root, so `velocity` imports

import pandas as pd
import plotly.graph_objects as go

from velocity import github_project, velocity as vel, burndown, points

items = github_project.fetch_items()
print(f"Fetched {len(items)} project items")

## Velocity
Average completed points per week over the last 3 weeks.

In [ ]:
v = vel.compute_velocity(items)  # last 3 completed weeks (vel.DEFAULT_VELOCITY_WEEKS)
recent = vel.completed_weeks(items)
print(f"Velocity: {v:.1f} points / week")
for wk, pts in recent:
    print(f"  week of {wk}: {pts} pts")

## Completed points per week

Full history through the last completed week. The **highlighted** last three
bars are the weeks averaged into the velocity; earlier weeks are muted for
context, and the current partial week is excluded.

In [ ]:
hist = burndown.completed_history(items)  # earliest data -> last completed week
velocity_weeks = {wk for wk, _ in vel.completed_weeks(items)}  # the weeks feeding velocity

HIGHLIGHT = "#4c78a8"             # the weeks that set the velocity
MUTED = "rgba(76,120,168,0.30)"   # earlier weeks: shown for context only
colors = [HIGHLIGHT if wk in velocity_weeks else MUTED for wk, _ in hist]

fig = go.Figure(go.Bar(
    x=[str(wk) for wk, _ in hist],
    y=[pts for _, pts in hist],
    marker_color=colors,
))
fig.add_hline(y=v, line_dash="dash", annotation_text=f"velocity {v:.1f}")
fig.update_layout(
    title="Completed points per week (highlighted weeks set the velocity)",
    xaxis_title="week", yaxis_title="points",
)
fig.show()

## Backlog burndown
Remaining backlog points projected forward at the current velocity, down to
zero. (Weekly *completed* points are shown in the chart above.) Dotted vertical
lines mark projected **release dates** — see the table below.

In [ ]:
proj = burndown.project_burndown(items, v)
total = burndown.backlog_total(items)
markers = burndown.release_marker_dates(burndown.backlog_items(items), v)

fig = go.Figure(go.Scatter(
    x=[wk.isoformat() for wk, _ in proj],
    y=[rem for _, rem in proj],
    mode="lines+markers",
    name="projected remaining",
))
for wk, it in markers:
    fig.add_vline(
        x=wk.isoformat(), line_dash="dot", line_color="#d9534f",
        annotation_text="🚀 " + points.marker_label(it.title),
        annotation_position="top",
    )
fig.update_xaxes(type="date")
finish = proj[-1][0] if len(proj) > 1 else None
title = f"Backlog burndown ({total} pts" + (f", done ~{finish})" if finish else ")")
fig.update_layout(title=title, xaxis_title="week", yaxis_title="points remaining")
fig.show()

## Projected completion — next 6 weeks

Backlog tasks grouped by the week they're projected to finish, in priority
order. **`~`** marks auto-estimated points (the task is unpointed); bugs count
as 0 points. 🚀 rows are release markers landing that week (they carry no points
of their own).

In [ ]:
import html
from IPython.display import HTML

backlog = burndown.backlog_items(items)
assignments = burndown.assign_tasks_to_weeks(backlog, v)  # includes release markers
grouped = burndown.upcoming_weeks(assignments, weeks=6)

STATUS_COLORS = github_project.fetch_status_colors()  # GitHub's own status colors
TYPE_COLORS = {"bug": "#d73a49", "feature": "#28a745", "task": "#6f42c1"}
POINTS_COLOR = "#e0a52a"  # gold: real points we set, readable on light and dark
ESTIMATE_COLOR = "#8a8a8a"  # medium gray: auto-estimated points, readable on either
RELEASE_COLOR = "#d9534f"  # release-marker rows
ROW_BORDER = "1px solid rgba(128,128,128,0.45)"  # medium gray, subtle in either mode


def _pill(text, bg):
    return (f'<span style="background:{bg};color:#fff;border-radius:3px;'
            f'padding:1px 6px;font-size:11px">{html.escape(text)}</span>')


def _task_row(it):
    pts = points.normalize(it)
    est = points.is_estimated(it)
    pts_label = f"~{pts}" if est else str(pts)
    pts_style = f"color:{ESTIMATE_COLOR if est else POINTS_COLOR};" + ("font-style:italic" if est else "")
    badges = _pill(it.status or "—", STATUS_COLORS.get(it.status, "#59636e"))
    typ = points.effective_type(it)
    if typ != points.DEFAULT_TYPE:  # hide the default "feature" badge (it's just noise)
        badges += "&nbsp;" + _pill(typ, TYPE_COLORS.get(typ, "#888"))
    title = html.escape(it.title)
    if it.url:
        # color:inherit -> follows the notebook theme (dark or light)
        title = f'<a href="{it.url}" style="color:inherit;text-decoration:none">{title}</a>'
    return (f'<tr style="border-top:{ROW_BORDER}">'
            f'<td style="padding:3px 8px;text-align:left;color:inherit">{title}</td>'
            f'<td style="padding:3px 8px;text-align:right;white-space:nowrap">{badges}</td>'
            f'<td style="width:44px;padding:3px 8px;text-align:right;{pts_style}">{pts_label}</td></tr>')


def _marker_row(it):
    label = html.escape(points.marker_label(it.title))
    if it.url:
        label = f'<a href="{it.url}" style="color:inherit;text-decoration:none">{label}</a>'
    return (f'<tr style="border-top:{ROW_BORDER}">'
            f'<td colspan="3" style="padding:4px 8px;color:{RELEASE_COLOR};font-weight:600">'
            f'🚀 {label} <span style="font-weight:normal;color:#888">— projected release</span></td></tr>')


def render(grouped):
    out = ['<div style="font-family:-apple-system,Segoe UI,sans-serif;max-width:780px">']
    out.append('<p style="color:#888;font-size:12px;margin:0 0 8px">'
               '<b>~</b> = auto-estimated points (task is unpointed) &nbsp;&middot;&nbsp; '
               'bugs count as 0 points &nbsp;&middot;&nbsp; '
               '<span style="color:#d9534f">🚀</span> = release marker</p>')
    for wk, group in grouped:
        tasks = [i for i in group if not points.is_release_marker(i)]
        markers = [i for i in group if points.is_release_marker(i)]
        total = sum(points.normalize(i) for i in tasks)
        label = f"{len(tasks)} task" + ("" if len(tasks) == 1 else "s")
        out.append(
            f'<h3 style="margin:16px 0 2px">Week of {wk:%b %d}'
            f'<span style="color:#888;font-weight:normal;font-size:13px"> &mdash; {label}, {total} pts</span></h3>')
        if not tasks and not markers:
            out.append('<div style="color:#888;font-size:13px;padding-left:6px">(nothing projected)</div>')
            continue
        out.append('<table style="border-collapse:collapse;width:100%;font-size:13px">')
        out.extend(_task_row(it) for it in tasks)
        out.extend(_marker_row(it) for it in markers)
        out.append("</table>")
    out.append("</div>")
    return HTML("".join(out))


render(grouped)

## Release markers

Projected completion date for each release marker (🚀 / ✅ / 🚢 in the title),
based on where it sits in the prioritized backlog. Markers carry no points of
their own.

In [ ]:
from IPython.display import HTML

markers = burndown.release_marker_dates(burndown.backlog_items(items), v)

if not markers:
    out = '<em style="color:#888">No release markers in the backlog.</em>'
else:
    rows = "".join(
        '<tr style="border-top:1px solid rgba(128,128,128,0.45)">'
        f'<td style="padding:4px 14px 4px 0;color:inherit">{points.marker_label(it.title)}</td>'
        f'<td style="padding:4px 0;text-align:right;color:#e0a52a;white-space:nowrap">{wk:%b %d, %Y}</td>'
        "</tr>"
        for wk, it in markers
    )
    out = (
        '<table style="border-collapse:collapse;font-size:13px;'
        'font-family:-apple-system,Segoe UI,sans-serif">'
        '<tr style="color:#888;font-size:12px;text-align:left">'
        '<th style="padding:0 14px 4px 0">release marker</th>'
        '<th style="padding:0 0 4px;text-align:right">projected date</th></tr>'
        f"{rows}</table>"
    )

HTML(out)